# Notebook 01 — Residue Manifold Construction

**Repo:** `residue-manifold-learning`  
**Purpose:** construct the baseline mod30 residue manifold, identify the eight prime residue lanes, and export reusable data/figures for later notebooks.

This notebook is intentionally simple: no ML, no CGCS metric, no sparse autoencoders yet. It defines the object that later notebooks learn from.


## Outputs

This notebook writes:

- `data/residues_mod30.csv`
- `data/primes_mod30.csv`
- `data/residue_lane_summary_mod30.csv`
- `figures/residue_histogram_mod30.png`
- `figures/residue_circle_mod30.png`
- `figures/residue_lane_matrix_mod30.png`


In [ ]:
# Standard imports
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sympy import primerange
except ImportError as exc:
    raise ImportError("This notebook needs sympy. Install with: pip install sympy") from exc

# Make imports work whether running from repo root or from notebooks/
import sys
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

DATA_DIR = repo_root / "data"
FIG_DIR = repo_root / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MOD = 30
N_MAX = 10_000

# Canonical ground-truth lanes for primes > 5 in Z/30Z.
# Later notebooks should import or reuse this exact ordering.
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
EXCLUDED_LANES_MOD30 = [r for r in range(MOD) if r not in VALID_LANES_MOD30]


## 1. Construct residue space

The baseline space is the finite cyclic residue system `Z / 30Z`. For each integer `n`, the residue lane is `n mod 30`.


In [ ]:
n_values = np.arange(N_MAX + 1)
residues = n_values % MOD

df_residues = pd.DataFrame({
    "n": n_values,
    "mod": MOD,
    "residue": residues,
})

df_residues.head()

## 2. Generate primes and prime residues

For primes above 5, all prime residues mod30 must be coprime to 30. This leaves eight possible lanes.


In [ ]:
primes = np.array(list(primerange(2, N_MAX + 1)), dtype=int)
prime_residues = primes % MOD

df_primes = pd.DataFrame({
    "p": primes,
    "mod": MOD,
    "residue": prime_residues,
    "is_small_prime_divisor_of_30": np.isin(primes, [2, 3, 5]),
})

# Lanes for primes larger than the small prime divisors of 30.
df_prime_lanes = df_primes.loc[~df_primes["is_small_prime_divisor_of_30"]].copy()
observed_lanes = sorted(df_prime_lanes["residue"].unique().tolist())
valid_lanes = VALID_LANES_MOD30.copy()

observed_lanes


In [ ]:
expected_lanes = [r for r in range(MOD) if math.gcd(r, MOD) == 1]
print("Canonical lanes:       ", VALID_LANES_MOD30)
print("Expected coprime lanes:", expected_lanes)
print("Observed prime lanes:  ", observed_lanes)
assert observed_lanes == expected_lanes == VALID_LANES_MOD30
print(f"Confirmed: {len(valid_lanes)} valid lanes out of {MOD} residues.")


## 3. Lane summary table

This table is useful for later notebooks. It separates all residue lanes into valid prime lanes and excluded lanes.


In [ ]:
# Explicit full-length count vector: 22 excluded residues are exactly zero.
counts_full = np.zeros(MOD, dtype=int)
lane_counts = df_prime_lanes["residue"].value_counts().sort_index()
for residue, count in lane_counts.items():
    counts_full[int(residue)] = int(count)

df_lane_summary = pd.DataFrame({
    "mod": MOD,
    "residue": np.arange(MOD),
    "gcd_residue_mod": [math.gcd(r, MOD) for r in range(MOD)],
    "is_coprime_lane": [r in VALID_LANES_MOD30 for r in range(MOD)],
    "prime_count_excluding_2_3_5": counts_full,
})

df_lane_summary["lane_label"] = np.where(
    df_lane_summary["is_coprime_lane"],
    "valid_prime_lane",
    "excluded_lane",
)

df_lane_summary


## 4. Density baseline

The mod30 prime lane density is the fraction of residue classes that remain possible after excluding residues sharing a factor with 30.


In [ ]:
lane_density = len(valid_lanes) / MOD
print(f"Valid lanes: {len(valid_lanes)} / {MOD}")
print(f"Lane density: {lane_density:.6f}")

## 5. Figure — prime residue histogram

This figure shows the eight valid lanes and the excluded lanes. It should be the simplest baseline plot for the paper introduction.


In [ ]:
# Use the explicit count vector so excluded lanes remain visible as zeros.
counts_all = pd.Series(counts_full, index=np.arange(MOD))

plt.figure(figsize=(10, 4.8))
plt.bar(counts_all.index.astype(str), counts_all.values)
plt.title("Prime Residue Lanes mod 30")
plt.xlabel("Residue class r mod 30")
plt.ylabel("Prime count up to N, excluding 2, 3, 5")
plt.xticks(rotation=0)
plt.tight_layout()

hist_path = FIG_DIR / "residue_histogram_mod30.png"
hist_clean_path = FIG_DIR / "residue_histogram_mod30_clean.png"
plt.savefig(hist_path, dpi=180, bbox_inches="tight")
plt.savefig(hist_clean_path, dpi=300, bbox_inches="tight")
plt.show()

hist_path, hist_clean_path


## 6. Figure — circular residue manifold

Embedding each residue class as an angle on the unit circle makes the mod30 structure visible.

This version explicitly sorts valid lanes by angular position before connecting them, then closes the loop across the 29 → 1 wraparound. That keeps the connected path cyclic rather than relying on list order.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 600)
all_residues = np.arange(MOD)
all_angles = 2 * np.pi * all_residues / MOD

valid_residues = np.array(VALID_LANES_MOD30)
valid_angles = 2 * np.pi * valid_residues / MOD

# Sort valid lanes by angular position before drawing the connected manifold path.
angle_order = np.argsort(valid_angles)
valid_residues_sorted = valid_residues[angle_order]
valid_angles_sorted = valid_angles[angle_order]
closed_valid_angles = np.r_[valid_angles_sorted, valid_angles_sorted[0]]

plt.figure(figsize=(7, 7))

# base circle
plt.plot(np.cos(theta), np.sin(theta), linewidth=1, alpha=0.4)

# all residues as small markers
plt.scatter(
    np.cos(all_angles),
    np.sin(all_angles),
    s=35,
    alpha=0.35,
    label="all residues",
)

# connected cyclic manifold path through valid prime lanes
plt.plot(
    np.cos(closed_valid_angles),
    np.sin(closed_valid_angles),
    linewidth=2,
    alpha=0.7,
    label="cyclic valid-lane path",
)

# valid lanes as larger markers
plt.scatter(
    np.cos(valid_angles_sorted),
    np.sin(valid_angles_sorted),
    s=120,
    label="valid prime lanes",
)

# residue labels
for r, a in zip(all_residues, all_angles):
    radius = 1.13 if r in VALID_LANES_MOD30 else 1.07
    plt.text(
        radius * np.cos(a),
        radius * np.sin(a),
        str(r),
        ha="center",
        va="center",
        fontsize=8,
    )

plt.title("Residue Manifold: valid prime lanes in Z/30Z")
plt.axis("equal")
plt.axis("off")
plt.legend(loc="upper right")
plt.tight_layout()

circle_path = FIG_DIR / "residue_circle_mod30.png"
circle_clean_path = FIG_DIR / "residue_circle_mod30_clean.png"
plt.savefig(circle_path, dpi=180, bbox_inches="tight")
plt.savefig(circle_clean_path, dpi=300, bbox_inches="tight")
plt.show()

print("Angular lane order:", valid_residues_sorted.tolist())
circle_path, circle_clean_path

## 7. Figure — lane indicator matrix

This compact matrix is useful later for comparing learned basis features against ground-truth residue lanes.


In [ ]:
lane_indicator = np.zeros((1, MOD), dtype=int)
lane_indicator[0, valid_lanes] = 1

plt.figure(figsize=(11, 1.8))
plt.imshow(lane_indicator, aspect="auto")
plt.yticks([0], ["valid lane"])
plt.xticks(range(MOD), range(MOD), fontsize=8)
plt.title("Ground-truth mod30 valid prime lane indicator")
plt.xlabel("Residue class r mod 30")
plt.tight_layout()

matrix_path = FIG_DIR / "residue_lane_matrix_mod30.png"
matrix_clean_path = FIG_DIR / "residue_lane_matrix_mod30_clean.png"
plt.savefig(matrix_path, dpi=180, bbox_inches="tight")
plt.savefig(matrix_clean_path, dpi=300, bbox_inches="tight")
plt.show()

matrix_path, matrix_clean_path


## 8. Save reusable data

Later notebooks should load these CSVs instead of reconstructing the baseline every time.


In [ ]:
residues_path = DATA_DIR / "residues_mod30.csv"
primes_path = DATA_DIR / "primes_mod30.csv"
lane_summary_path = DATA_DIR / "residue_lane_summary_mod30.csv"

df_residues.to_csv(residues_path, index=False)
df_primes.to_csv(primes_path, index=False)
df_lane_summary.to_csv(lane_summary_path, index=False)

print(residues_path)
print(primes_path)
print(lane_summary_path)

## 9. Notebook 01 claim

Prime numbers under mod30 occupy a fixed set of eight coprime residue lanes:

`1, 7, 11, 13, 17, 19, 23, 29`

The remaining 22 residue classes have exactly zero prime counts after excluding the small prime divisors `2, 3, 5`. This creates a discrete residue manifold: a constrained sampling space for downstream learning.

Tang-style bridge:

> Structure precedes speed. Here, residue constraints define the sampling manifold before any learning occurs.


## 10. Optional output bundle

Run this cell to prepare a zip file containing generated `data/` and `figures/` outputs. Uncomment the final two lines when using Colab and you want the browser download to trigger.

In [ ]:
# --- Optional: Download Notebook 01 outputs: data + figures ---

import os
import zipfile

zip_name = "01_residue_space_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)